# 01 - Data Collection

Scrapes stock specs from parkers.co.uk and modification figures from goapr.com.

Outputs (all in `data/raw/`):
- `cars_raw.csv` - one row per stock car variant
- `modifications_raw.csv` - one row per stage/car combination from APR
- `builds_raw.csv` - one row per real modded car (manual entry)

No database needed - everything stays as CSVs.

In [5]:
import sys
sys.path.insert(0, '..')  # lets Python find src/ when running from notebooks/

import pandas as pd
from src.scraper import scrape_parkers, scrape_parkers_batch, scrape_goapr_batch, save_csv, append_csv

---
## 1. Stock specs - parkers.co.uk

How to find the right URL:
1. Go to parkers.co.uk
2. Search for the car
3. Click into the exact variant (engine, trim, body style)
4. Click Specs in the tab bar
5. Copy the URL - it ends in `/specs/`

In [3]:
CAR_URLS = [
    # Ford Focus ST Mk3 2.0T Estate
    "https://www.parkers.co.uk/ford/focus/st-2012/20t-st-3-estate-(0115-)-5d/specs/",

    # Add your other cars here — paste the /specs/ URL directly from parkers
    # "https://www.parkers.co.uk/volkswagen/golf/..../specs/",
    # "https://www.parkers.co.uk/honda/civic-type-r/..../specs/",
]

cars_df = scrape_parkers_batch(CAR_URLS)
cars_df

INFO  Fetching https://www.parkers.co.uk/ford/focus/st-2012/20t-st-3-estate-(0115-)-5d/specs/


,source_url,raw_title,hp_stock,torque_nm_stock,weight_kg,zero_to_100,top_speed_kph,top_speed_mph,engine_cc,drivetrain,transmission,fuel_type,cylinders,co2_gkm,_raw
0,https://www.parkers.co.uk/ford/focus/st-2012/2...,Ford Focus ST2.0T ST-3 Estate (01/15-) 5d Spec...,246.0,345.0,1461.0,6.5,248,154.0,1999.0,Front wheel drive,Manual,Petrol,4.0,159.0,{'available new from': 'January 2015 - Februar...


In [ ]:
# run this if any column shows NaN — it prints every label the scraper found on the page.
# if an expected value is missing, the parkers label probably differs slightly.

if not cars_df.empty:
    print(cars_df.iloc[0]['_raw'])

In [6]:
cars_clean = cars_df.drop(columns=['_raw'], errors='ignore')
save_csv(cars_clean, '../data/raw/cars_raw.csv')
cars_clean

INFO  Saved 1 rows → ../data/raw/cars_raw.csv


,source_url,raw_title,hp_stock,torque_nm_stock,weight_kg,zero_to_100,top_speed_kph,top_speed_mph,engine_cc,drivetrain,transmission,fuel_type,cylinders,co2_gkm
0,https://www.parkers.co.uk/ford/focus/st-2012/2...,Ford Focus ST2.0T ST-3 Estate (01/15-) 5d Spec...,246.0,345.0,1461.0,6.5,248,154.0,1999.0,Front wheel drive,Manual,Petrol,4.0,159.0


---
## 2. Modifications - goapr.com

APR publish exact before/after hp and torque for each ECU tune.
All figures are wheel hp (whp) from their in-house dyno.

In [7]:
APR_URLS = [
    "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html",
    "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is38.html",
    # add Revo / Mountune / Unitronic URLs here
]

mods_df = scrape_goapr_batch(APR_URLS)
mods_df

INFO  Fetching https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html
ERROR  Failed https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html: 403 Client Error: Forbidden for url: https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html
INFO  Fetching https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is38.html
ERROR  Failed https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is38.html: 403 Client Error: Forbidden for url: https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is38.html


""


In [ ]:
save_csv(mods_df, '../data/raw/modifications_raw.csv')

---
## 3. Builds - manual entry

Forum and YouTube builds need manual entry (login walls, JavaScript).
Add rows as you find them - each run appends without overwriting.

`mod_description` = copy the exact text from the post, don't rephrase  
`hp_type` = `whp` (forum dynos) / `crank` (manufacturer claims) / `unknown`  
`confidence`: 0.90 tuner dyno · 0.75 forum dyno with image · 0.60 forum no image · 0.35 estimate

In [ ]:
MANUAL_BUILDS = [
    {
        "car_id":           "ford_focus_st_mk3",
        "mod_description":  "APR Stage 1 ECU tune only, stock hardware",
        "result_hp":        307,
        "result_torque_nm": None,
        "result_0_100":     None,
        "result_top_speed": None,
        "measurement_type": "dyno",
        "hp_type":          "whp",
        "source_type":      "tuner_site",
        "confidence":       0.90,
        "source_url":       "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html",
    },
    # add more rows here
]

builds_df = append_csv(MANUAL_BUILDS, '../data/raw/builds_raw.csv')
builds_df

---
## 4. Sanity check

In [ ]:
for name, path in [
    ("cars",          '../data/raw/cars_raw.csv'),
    ("modifications", '../data/raw/modifications_raw.csv'),
    ("builds",        '../data/raw/builds_raw.csv'),
]:
    try:
        df = pd.read_csv(path)
        print(f"{name:15} {len(df):3} rows  |  {list(df.columns)}")
    except FileNotFoundError:
        print(f"{name:15} not created yet")